# Base Python File

In [1]:
# Import Packages
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline
import seaborn as sns

In [2]:
# Create timestamp and author
from datetime import datetime
time_now=datetime.now().strftime('%m_%d_%Y_%I%M%p')
print("Current Time:{}".format(time_now))
print ("Nhi Bui")

Current Time:09_28_2026_0421AM
Nhi Bui


In [3]:
# Read the dataset and store it as data2026
data2026=pd.read_csv('2026PartYear.csv',encoding='utf-8')

In [6]:
#Read in the 2025 dataset as data2025

# Read the dataset and store it as data2025
data2025=pd.read_csv('2025.csv',encoding='utf-8')

In [11]:
print("2025 columns:")
print(data2025.columns.tolist())

print("\n2026 columns:")
print(data2026.columns.tolist())

display(data2025.head(3))
display(data2026.head(3))

2025 columns:
['Report', 'Timeframe', 'Period', 'Type', 'Geography', 'Current Year Week Ending', 'ASP Current Year', 'Total Bulk and Bags', '4046', '4225', '4770', 'TotalBagged', 'Sml Bagged', 'Lrg Bagged', 'X-Lrg Bagged', 'Unknown Bag Size']

2026 columns:
['Report', 'Timeframe', 'Period', 'Type', 'Geography', 'Current Year Week Ending', 'ASP Current Year', 'Total Bulk and Bags', '4046', '4225', '4770', 'TotalBagged', 'Sml Bagged', 'Lrg Bagged', 'X-Lrg Bagged', 'Unknown Bag Size']


,Report,Timeframe,Period,Type,Geography,Current Year Week Ending,ASP Current Year,Total Bulk and Bags,4046,4225,4770,TotalBagged,Sml Bagged,Lrg Bagged,X-Lrg Bagged,Unknown Bag Size
0,Category,Weekly,52,Conventional,5 Boroughs/New York City,2025/12/28,1.095919,584086.7021,98699.86337,169066.4447,1581.867365,314738.5267,284656.3211,4644.746432,6223.100331,19214.35889
1,Category,Weekly,51,Conventional,5 Boroughs/New York City,2025/12/21,1.115703,743525.7741,118644.87660,308259.2727,2684.404897,313937.2200,280796.1232,7056.000000,9868.904359,16216.19243
2,Category,Weekly,50,Conventional,5 Boroughs/New York City,2025/12/14,1.008697,857684.6458,126587.72660,344061.2502,1666.221558,385369.4475,362029.4992,6871.200000,3716.701695,12752.04659


,Report,Timeframe,Period,Type,Geography,Current Year Week Ending,ASP Current Year,Total Bulk and Bags,4046,4225,4770,TotalBagged,Sml Bagged,Lrg Bagged,X-Lrg Bagged,Unknown Bag Size
0,Category,Weekly,12,Conventional,5 Boroughs/New York City,2026/03/22,1.079701,861791.6216,141926.1937,325330.2392,3597.595971,390937.5927,339735.8554,27441.99605,6502.811928,17256.92930
1,Category,Weekly,11,Conventional,5 Boroughs/New York City,2026/03/15,1.061474,957397.8336,147209.5073,386184.7090,2969.438726,421034.1786,371778.4086,18890.83033,11987.800280,18377.13941
2,Category,Weekly,10,Conventional,5 Boroughs/New York City,2026/03/08,1.046660,919553.7267,154227.6405,338720.0493,4881.431601,421724.6054,385392.9832,16913.15632,5779.667905,13638.79796


In [16]:
# Set column Names
geography_col = "Geography"
category_col = "Type"
date_col = "Current Year Week Ending"
period_col = "Period"

volume_col = "Total Bulk and Bags"
price_col = "ASP Current Year"
bagged_col = "TotalBagged"
large_bagged_col = "Lrg Bagged"
code_4225_col = "4225"
code_4046_col = "4046"

us_label = "Total U.S."
conventional_label = "Conventional"
organic_label = "Organic"

print("Geography labels containing 'Total':")
print(data2025.loc[
    data2025[geography_col].astype(str).str.contains("Total", case=False),
    geography_col
].unique())

print("\nAvocado types:")
print(data2025[category_col].unique())

print("\n2026 periods:")
print(sorted(data2026[period_col].dropna().unique()))

Geography labels containing 'Total':
['Total U.S.']

Avocado types:
['Conventional' 'Organic']

2026 periods:
[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32]


In [17]:
#Select U.S. weeks 1–32 in each file
def prepare(data, year):
    df = data.copy()

    df = df[
        (df["Geography"] == "Total U.S.")
        & (df["Type"].isin(["Conventional", "Organic"]))
    ].copy()

    df["Period"] = pd.to_numeric(df["Period"], errors="coerce")
    df = df[df["Period"].between(1, 32)].copy()

    df["Current Year Week Ending"] = pd.to_datetime(
        df["Current Year Week Ending"], errors="coerce"
    )

    number_columns = [
        "ASP Current Year", "Total Bulk and Bags",
        "TotalBagged", "Lrg Bagged", "4225", "4046"
    ]

    for column in number_columns:
        df[column] = pd.to_numeric(
            df[column].astype(str).str.replace(r"[$,]", "", regex=True),
            errors="coerce"
        )

    df["year"] = year
    return df

national = pd.concat(
    [prepare(data2025, 2025), prepare(data2026, 2026)],
    ignore_index=True
)

display(national.groupby(["year", "Type"]).agg(
    rows=("Period", "size"),
    weeks=("Period", "nunique"),
    first_date=("Current Year Week Ending", "min"),
    last_date=("Current Year Week Ending", "max")
))


rows  weeks first_date  last_date
year Type                                           
2025 Conventional    32     32 2025-01-05 2025-08-10
     Organic         32     32 2025-01-05 2025-08-10
2026 Conventional    32     32 2026-01-04 2026-08-09
     Organic         32     32 2026-01-04 2026-08-09

In [18]:
#Check Volumes and Weighted Average Prices
def weighted_price(group):
    valid = group.dropna(subset=["ASP Current Year", "Total Bulk and Bags"])
    valid = valid[valid["Total Bulk and Bags"] > 0]

    return np.average(
        valid["ASP Current Year"],
        weights=valid["Total Bulk and Bags"]
    )

rows = []

for (year, avocado_type), group in national.groupby(["year", "Type"]):
    rows.append({
        "year": year,
        "Type": avocado_type,
        "volume": group["Total Bulk and Bags"].sum(),
        "weighted_price": weighted_price(group),
        "bagged": group["TotalBagged"].sum(),
        "large_bagged": group["Lrg Bagged"].sum(),
        "4225": group["4225"].sum(),
        "4046": group["4046"].sum()
    })

summary = pd.DataFrame(rows)
display(summary)

,year,Type,volume,weighted_price,bagged,large_bagged,4225,4046
0,2025,Conventional,1.764839e+09,1.215428,6.965073e+08,6.104266e+07,1.878611e+08,8.565450e+08
1,2025,Organic,1.058595e+08,1.740676,8.580812e+07,1.924889e+05,9.416399e+06,1.061538e+07
2,2026,Conventional,2.020685e+09,0.979122,8.342960e+08,9.544323e+07,2.578938e+08,8.790437e+08
3,2026,Organic,1.051156e+08,1.561718,8.545761e+07,8.847774e+05,9.634050e+06,1.000733e+07


In [22]:
#Check Percentage Changes and Shares
def comparison(measure, include_total=False):
    table = summary.pivot(index="Type", columns="year", values=measure)

    if include_total:
        table.loc["Total"] = table.sum()

    table["% change"] = (table[2026] / table[2025] - 1) * 100
    return table.round(3)

print("VOLUME")
display(comparison("volume", include_total=True))

print("VOLUME-WEIGHTED AVERAGE PRICE")
display(comparison("weighted_price"))

print("BAGGED VOLUME")
display(comparison("bagged", include_total=True))

print("LARGE BAGGED VOLUME")
display(comparison("large_bagged"))

print("4225 VOLUME")
display(comparison("4225"))

print("4046 VOLUME")
display(comparison("4046"))

VOLUME


year,2025,2026,% change
Type,,,
Conventional,1.764839e+09,2.020685e+09,14.497
Organic,1.058595e+08,1.051156e+08,-0.703
Total,1.870699e+09,2.125800e+09,13.637


VOLUME-WEIGHTED AVERAGE PRICE


year,2025,2026,% change
Type,,,
Conventional,1.215,0.979,-19.442
Organic,1.741,1.562,-10.281


BAGGED VOLUME


year,2025,2026,% change
Type,,,
Conventional,6.965073e+08,8.342960e+08,19.783
Organic,8.580812e+07,8.545761e+07,-0.408
Total,7.823154e+08,9.197537e+08,17.568


... (2 outputs left out)

4225 VOLUME


year,2025,2026,% change
Type,,,
Conventional,1.878611e+08,2.578938e+08,37.279
Organic,9.416399e+06,9.634050e+06,2.311


4046 VOLUME


year,2025,2026,% change
Type,,,
Conventional,8.565450e+08,8.790437e+08,2.627
Organic,1.061538e+07,1.000733e+07,-5.728


In [24]:
# Calculate Revenue by multiply ASP to total bulk and bags
for df in [data2025, data2026]:
    df["Calculated Revenue"] = (
        pd.to_numeric(df["ASP Current Year"], errors="coerce")
        * pd.to_numeric(df["Total Bulk and Bags"], errors="coerce")
    )

display(data2025[
    ["ASP Current Year", "Total Bulk and Bags", "Calculated Revenue"]
].head())

,ASP Current Year,Total Bulk and Bags,Calculated Revenue
0,1.095919,584086.7021,640111.976734
1,1.115703,743525.7741,829553.991762
2,1.008697,857684.6458,865143.923161
3,1.056224,858937.0556,907229.947216
4,1.187256,569309.2162,675915.881849


In [25]:
def calculate_shares(data, year):
    # Match the memo's scope: Total U.S., weeks 1–32.
    period = pd.to_numeric(data["Period"], errors="coerce")
    df = data.loc[
        (data["Geography"] == "Total U.S.")
        & period.between(1, 32)
        & data["Type"].isin(["Conventional", "Organic"])
    ].copy()

    for column in ["Total Bulk and Bags", "TotalBagged"]:
        df[column] = pd.to_numeric(df[column], errors="coerce")

    totals = df.groupby("Type")[["Total Bulk and Bags", "TotalBagged"]].sum()

    conventional_volume = totals.loc["Conventional", "Total Bulk and Bags"]
    organic_volume = totals.loc["Organic", "Total Bulk and Bags"]

    return {
        "Year": year,
        "Organic share of total volume (%)":
            organic_volume / (conventional_volume + organic_volume) * 100,
        "Bagged share of conventional volume (%)":
            totals.loc["Conventional", "TotalBagged"]
            / conventional_volume * 100,
        "Bagged share of organic volume (%)":
            totals.loc["Organic", "TotalBagged"]
            / organic_volume * 100
    }

checks = pd.DataFrame([
    calculate_shares(data2025, 2025),
    calculate_shares(data2026, 2026)
]).set_index("Year")

display(checks.round(2))

,Organic share of total volume (%),Bagged share of conventional volume (%),Bagged share of organic volume (%)
Year,,,
2025,5.66,39.47,81.06
2026,4.94,41.29,81.30
